# 01a. QuartzNet + InterCTC — вспомогательный промежуточный CTC-loss

## Идея

Модель выдаёт `EncoderOutput.intermediate` — активации после блока B2 (256-канальный тензор `[B, T', 256]`).
Мы проецируем их линейным слоем `InterCTCHead` в пространство vocab и добавляем CTC-loss
на этот промежуточный выход с весом `inter_ctc_weight`.

Это заставляет энкодер уже на средних слоях формировать семантически значимые представления
и работает как регуляризация — аналогично методу Lee & Watanabe (Intermediate Loss Regularization, 2020).

**Адаптация под Wave-1 Batch:** `Batch` содержит 6 полей (`audio`, `audio_lengths`, `targets`,
`target_lengths`, `spk_ids`, `transcriptions`). `InterCTCHead` использует `out.intermediate`
из `EncoderOutput` — это поле не было удалено.

In [ ]:
import sys
import os

# Добавляем src/ в путь, если запуск из папки notebooks/experiments/
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "..", "src"))

from gp1.env import setup_environment
paths, device = setup_environment()
print(f"Platform OK. Device: {device}")
print(f"Train CSV: {paths.train_csv}")

In [ ]:
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

from gp1.data.manifest import records_from_csv, leave_n_speakers_out_split
from gp1.data.dataset import SpokenNumbersDataset
from gp1.data.collate import collate_fn
from gp1.data.audio_aug import AudioAugmenter
from gp1.data.spec_aug import SpecAugmenter
from gp1.features.melbanks import LogMelFilterBanks
from gp1.losses.ctc import CTCLoss
from gp1.models.quartznet import QuartzNet10x4
from gp1.train.optim import build_novograd
from gp1.train.schedulers import build_cosine_warmup
from gp1.train.checkpoint import save_best
from gp1.train.metrics import compute_cer, compute_per_speaker_cer
from gp1.decoding.greedy import greedy_decode
from gp1.text.vocab import CharVocab
from gp1.text.normalize import digits_to_words
from gp1.types import AugConfig

In [ ]:
class InterCTCHead(nn.Module):
    """Auxiliary CTC head applied to intermediate encoder features.

    Adapted for Wave-1 Batch: uses out.intermediate (not a separate field).

    Args:
        d_mid: Dimension of intermediate encoder features (256 for QuartzNet10x4).
        vocab_size: Output vocabulary size (same as main head).
        blank_id: CTC blank token id (default 0).
    """

    def __init__(self, d_mid: int, vocab_size: int, blank_id: int = 0) -> None:
        super().__init__()
        self.proj = nn.Linear(d_mid, vocab_size)
        self._ctc = CTCLoss(blank_id=blank_id)

    def forward(
        self,
        mid_features: torch.Tensor,    # [B, T, d_mid]
        input_lengths: torch.Tensor,   # [B]
        targets: torch.Tensor,         # [B, U]
        target_lengths: torch.Tensor,  # [B]
    ) -> torch.Tensor:
        assert mid_features.dim() == 3, "mid_features must be [B, T, D]"
        logits = self.proj(mid_features)
        log_probs = F.log_softmax(logits.float(), dim=-1)
        return self._ctc(log_probs, targets, input_lengths, target_lengths)

In [ ]:
# --- Manifests + vocab ---
all_records = records_from_csv(paths.train_csv, paths.train_root)
dev_records = records_from_csv(paths.dev_csv, paths.dev_root)
vocab = CharVocab()
print(f"Train: {len(all_records)} records. Dev: {len(dev_records)} records.")
print(f"Vocab size: {vocab.vocab_size}, blank_id: {vocab.blank_id}")

In [ ]:
# --- Фиксированные гиперпараметры (не перебираются) ---
HP_FIXED = dict(
    samplerate=16000,
    n_mels=80,
    n_fft=512,
    hop_length=160,
    win_length=400,
)

# --- Настраиваемые HP (один трайл, фиксированные значения) ---
HP = dict(
    lr=0.01,
    weight_decay=1e-3,
    dropout=0.1,
    d_model=256,
    subsample_factor=2,
    warmup_steps=3000,
    inter_ctc_weight=0.3,
    max_epochs=30,
    grad_accum=2,
    batch_size=32,
    speed_prob=0.5,
    gain_prob=0.7,
    noise_prob=0.0,
    freq_mask_param=15,
    time_mask_param=25,
)
print("HP:", HP)

In [ ]:
# --- DataLoaders ---
aug_cfg = AugConfig(
    speed_prob=HP["speed_prob"],
    gain_prob=HP["gain_prob"],
    noise_prob=HP["noise_prob"],
    seed=42,
)
train_augmenter = AudioAugmenter(aug_cfg)

train_ds = SpokenNumbersDataset(
    records=all_records,
    vocab=vocab,
    target_samplerate=HP_FIXED["samplerate"],
    augmenter=train_augmenter,
)
dev_ds = SpokenNumbersDataset(
    records=dev_records,
    vocab=vocab,
    target_samplerate=HP_FIXED["samplerate"],
    augmenter=None,
)

train_loader = DataLoader(
    train_ds, batch_size=HP["batch_size"], shuffle=True,
    num_workers=2, collate_fn=collate_fn, pin_memory=True,
)
dev_loader = DataLoader(
    dev_ds, batch_size=HP["batch_size"], shuffle=False,
    num_workers=2, collate_fn=collate_fn,
)
print(f"Train batches: {len(train_loader)}, Dev batches: {len(dev_loader)}")

In [ ]:
# --- Mel extractor + SpecAugment ---
mel_extractor = LogMelFilterBanks(
    n_fft=HP_FIXED["n_fft"],
    samplerate=HP_FIXED["samplerate"],
    hop_length=HP_FIXED["hop_length"],
    win_length=HP_FIXED["win_length"],
    n_mels=HP_FIXED["n_mels"],
).to(device)

spec_aug = SpecAugmenter(
    freq_mask_param=HP["freq_mask_param"],
    num_freq_masks=2,
    time_mask_param=HP["time_mask_param"],
    num_time_masks=5,
    seed=42,
)

## Развёрнутый тренировочный цикл (без trainer.fit)

Ниже — полный цикл с явной точкой внедрения `InterCTCHead`. Студент видит, где суммируются два loss'а.

**Best practice — добавлен `torch.nn.utils.clip_grad_norm_`** с `max_norm=1.0` перед каждым шагом оптимизатора.
Это предотвращает взрывной рост градиентов при суммировании двух loss'ов (main CTC + inter CTC).
Значение `max_norm=1.0` — стандартное для CTC-задач с NovoGrad.

In [ ]:
# --- Модель + inter_ctc_head ---
model = QuartzNet10x4(
    vocab_size=vocab.vocab_size,
    d_model=HP["d_model"],
    dropout=HP["dropout"],
    subsample_factor=HP["subsample_factor"],
).to(device)

# model._d_mid == 256 (B2-block output channels, tapped before B3)
inter_head = InterCTCHead(
    d_mid=model._d_mid,
    vocab_size=vocab.vocab_size,
    blank_id=vocab.blank_id,
).to(device)

ctc_loss_fn = CTCLoss(blank_id=vocab.blank_id)

all_params = list(model.parameters()) + list(inter_head.parameters())
optimizer = build_novograd(
    all_params,
    lr=HP["lr"],
    betas=(0.95, 0.5),
    weight_decay=HP["weight_decay"],
)
total_steps = len(train_loader) * HP["max_epochs"]
scheduler = build_cosine_warmup(
    optimizer, total_steps=total_steps, warmup_steps=HP["warmup_steps"]
)

ckpt_dir = paths.ckpt_root / "01a_inter_ctc"
best_cer = float("inf")
history = []
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")
print(f"InterCTCHead params: {sum(p.numel() for p in inter_head.parameters()):,}")

In [ ]:
for epoch in tqdm(range(HP["max_epochs"]), desc="epochs"):
    # ---- Train ----
    model.train()
    inter_head.train()
    spec_aug.train()
    grad_step = 0
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(train_loader, desc=f"epoch {epoch}", leave=False)):
        audio = batch.audio.to(device)
        audio_lengths = batch.audio_lengths.to(device)
        targets = batch.targets.to(device)
        target_lengths = batch.target_lengths.to(device)

        # Mel features (no grad needed for the frontend)
        with torch.no_grad():
            mel = mel_extractor(audio)

        # mel_lengths: map audio sample lengths -> frame counts
        mel_lengths = (
            (audio_lengths + HP_FIXED["hop_length"] - 1) // HP_FIXED["hop_length"]
        ).clamp(max=mel.size(-1))

        mel = spec_aug(mel, mel_lengths)

        # Forward pass with fp16 autocast
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
            out = model(mel, mel_lengths)

        # Loss computation in fp32 (CTC is numerically unstable in fp16)
        with torch.autocast(device_type=device.type, enabled=False):
            loss_main = ctc_loss_fn(
                out.log_probs.float(), targets, out.output_lengths, target_lengths
            )
            # --- InterCTC loss: point of injection ---
            if out.intermediate is not None:
                loss_inter = inter_head(
                    out.intermediate, out.output_lengths, targets, target_lengths
                )
            else:
                loss_inter = torch.tensor(0.0, device=device)

        loss = loss_main + HP["inter_ctc_weight"] * loss_inter

        # Gradient accumulation
        (loss / HP["grad_accum"]).backward()
        grad_step += 1
        if grad_step % HP["grad_accum"] == 0:
            torch.nn.utils.clip_grad_norm_(all_params, max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    # ---- Validation ----
    model.eval()
    inter_head.eval()
    spec_aug.eval()
    all_refs, all_hyps, all_spks = [], [], []

    with torch.no_grad():
        for batch in dev_loader:
            audio = batch.audio.to(device)
            audio_lengths = batch.audio_lengths.to(device)
            mel = mel_extractor(audio)
            mel_lengths = (
                (audio_lengths + HP_FIXED["hop_length"] - 1) // HP_FIXED["hop_length"]
            ).clamp(max=mel.size(-1))
            out = model(mel, mel_lengths)
            hyps = greedy_decode(out.log_probs, out.output_lengths, vocab)
            all_refs.extend(batch.transcriptions)
            all_hyps.extend(hyps)
            all_spks.extend(batch.spk_ids)

    ref_words = [digits_to_words(r) for r in all_refs]
    val_cer = compute_cer(ref_words, all_hyps)
    spk_cer = compute_per_speaker_cer(ref_words, all_hyps, all_spks)
    max_spk_cer = max(spk_cer.values()) if spk_cer else 0.0

    history.append({"epoch": epoch, "val_cer": val_cer, "max_spk_cer": max_spk_cer})
    tqdm.write(f"epoch {epoch:3d} | val_cer={val_cer:.4f} | max_spk_cer={max_spk_cer:.4f}")

    if val_cer < best_cer:
        best_cer = val_cer
        ckpt_path = save_best(
            model,
            meta=dict(arch="quartznet10x4", hparams=HP, val_cer=val_cer, epoch=epoch),
            ckpt_dir=ckpt_dir,
        )
        tqdm.write(f"  Checkpoint saved: {ckpt_path}")

## Итог

In [ ]:
import pandas as pd

df = pd.DataFrame(history)
print(f"Best val CER: {best_cer:.4f}")
print(f"Checkpoint dir: {ckpt_dir}")
print()
print(df.tail(10).to_string(index=False))